# CLEAR-Face — Milestone 2: FER Training (Kaggle GPU)

Self-contained Kaggle notebook. It rebuilds the **same 5-way split** (seed 42)
directly from the attached AffectNet dataset, so no Windows paths are needed.

**Before running:**
1. Add data → `antruong2477/affectnet-dataset`  (mounts at `/kaggle/input/affectnet-dataset/`)
2. Right panel → **Accelerator = GPU**, **Internet = On** (needed once for pretrained weights)
3. Run all cells.

Split: train 36% · val 9% · test 15% · care_dev 15% · final_test 25%
(only train/val/test are used here; care_dev & final_test are held out).


In [ ]:
# Cell 1 — Imports & device
import os, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from PIL import Image
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device      :", device)
print("PyTorch     :", torch.__version__)              # fixed: was torch.version
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "No GPU! Set Accelerator = GPU in the right panel."


In [ ]:
# Cell 2 — Locate dataset and scan images
# AffectNet has no participant IDs -> each image is its own participant.
CLASSES = ["angry","disgust","fear","happy","neutral","sad","surprise"]
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASSES)}
IMAGE_EXTS = {".jpg",".jpeg",".png",".bmp",".webp"}

# AffectNet numeric folders (some mirrors use these) -> class name. 7=contempt dropped.
AFFECTNET_INDEX = {0:"neutral",1:"happy",2:"sad",3:"surprise",4:"fear",5:"disgust",6:"angry",7:None}

def norm_label(name):
    s = str(name).strip().lower()
    if s.isdigit():
        return AFFECTNET_INDEX.get(int(s))
    return s if s in CLASS_TO_IDX else None

# Auto-detect the dataset root under /kaggle/input
def find_root():
    base = Path("/kaggle/input")
    for child in sorted(base.iterdir()):
        if not child.is_dir():
            continue
        for sub in child.rglob("*"):
            if sub.is_dir() and norm_label(sub.name) is not None:
                return child
    raise FileNotFoundError("Could not find a class folder under /kaggle/input. Add the dataset.")

DATA_ROOT = find_root()
print("Dataset root:", DATA_ROOT)

rows = []
for p in sorted(DATA_ROOT.rglob("*")):
    if p.suffix.lower() not in IMAGE_EXTS or not p.is_file():
        continue
    label = None
    for parent in p.parents:
        if parent == DATA_ROOT.parent:
            break
        label = norm_label(parent.name)
        if label is not None:
            break
    if label is not None:
        rows.append({"image_path": str(p), "label": label})

df = pd.DataFrame(rows)
print(f"Found {len(df):,} labelled images across {df['label'].nunique()} classes")
print(df["label"].value_counts().to_string())


In [ ]:
# Cell 3 — 5-way stratified split (identical logic & seed to build_manifest_full.py)
FINAL_TEST_FRAC = 0.25
CARE_DEV_FRAC   = 0.15
_FER            = 1.0 - FINAL_TEST_FRAC - CARE_DEV_FRAC   # 0.60
FER_TRAIN_FRAC  = 0.60 * _FER   # 0.36
FER_VAL_FRAC    = 0.15 * _FER   # 0.09
FER_TEST_FRAC   = 0.25 * _FER   # 0.15

def five_way_split(df, seed=42):
    rng = np.random.default_rng(seed)
    out = pd.Series(index=df.index, dtype=object)
    for label, g in df.groupby("label"):
        idx = g.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx)
        n_ft  = max(1, round(n*FINAL_TEST_FRAC))
        n_cd  = max(1, round(n*CARE_DEV_FRAC))
        n_te  = max(1, round(n*FER_TEST_FRAC))
        n_va  = max(1, round(n*FER_VAL_FRAC))
        n_tr  = n - n_ft - n_cd - n_te - n_va
        if n_tr < 1:
            out[idx] = "train"; continue
        ptr = 0
        out[idx[ptr:ptr+n_tr]] = "train";      ptr += n_tr
        out[idx[ptr:ptr+n_va]] = "val";        ptr += n_va
        out[idx[ptr:ptr+n_te]] = "test";       ptr += n_te
        out[idx[ptr:ptr+n_cd]] = "care_dev";   ptr += n_cd
        out[idx[ptr:ptr+n_ft]] = "final_test"
    return out

# Deterministic ordering -> stable IDs
df = df.sort_values(["label","image_path"]).reset_index(drop=True)
df["split"] = five_way_split(df, seed=SEED)
df["label_idx"] = df["label"].map(CLASS_TO_IDX)
w = max(6, len(str(len(df))))
df["image_id"] = [f"img_{i:0{w}d}" for i in range(1, len(df)+1)]
df["participant_id"] = df["image_id"]
df["source_dataset"] = "affectnet"

print(df.groupby(["split","label"]).size().unstack(fill_value=0)
        .reindex(["train","val","test","care_dev","final_test"]).to_string())
print()
print(df["split"].value_counts(normalize=True).mul(100).round(1).to_string())

# Leakage check
assert df.groupby("image_id")["split"].nunique().max() == 1, "LEAKAGE!"
print("\nLeakage check passed.")

# Save manifest + label map to /kaggle/working (downloadable, GitHub-safe text)
os.makedirs("/kaggle/working/metadata", exist_ok=True)
df[["image_id","image_path","participant_id","source_dataset","split","label"]] \
  .to_csv("/kaggle/working/metadata/dataset_manifest.csv", index=False)
with open("/kaggle/working/metadata/label_map.json","w") as f:
    json.dump(CLASS_TO_IDX, f, indent=2)
print("Saved manifest + label_map to /kaggle/working/metadata/")

train_df = df[df["split"]=="train"].reset_index(drop=True)
val_df   = df[df["split"]=="val"].reset_index(drop=True)
test_df  = df[df["split"]=="test"].reset_index(drop=True)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")


In [ ]:
# Cell 4 — Dataset, transforms, dataloaders
class FERDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.df = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.loc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["label_idx"])

MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

BATCH = 64   # GPU can handle a bigger batch than CPU
train_loader = DataLoader(FERDataset(train_df, train_tf), batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(FERDataset(val_df,   eval_tf),  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print("Loaders ready. Batches:", len(train_loader), len(val_loader))


In [ ]:
# Cell 5 — Build models (Primary = EfficientNet-B2, Comparison = MobileNetV3)
def build_finetune_model(model_name, num_classes=7, freeze_backbone=True):
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
        for p in model.get_classifier().parameters():
            p.requires_grad = True
    return model.to(device)

primary_model    = build_finetune_model("efficientnet_b2",        freeze_backbone=True)
comparison_model = build_finetune_model("mobilenetv3_large_100",  freeze_backbone=True)

def n_train(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
def n_total(m): return sum(p.numel() for p in m.parameters())
print(f"EfficientNet-B2 trainable={n_train(primary_model):,} total={n_total(primary_model):,}")
print(f"MobileNetV3     trainable={n_train(comparison_model):,} total={n_total(comparison_model):,}")


In [ ]:
# Cell 6 — Training helpers (optimizer, train/val, progressive unfreeze)
def get_optimizer_and_scheduler(model, lr=1e-3):
    params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    return optimizer, scheduler

def train_one_epoch(model, loader, optimizer, criterion):
    model.train(); run_loss=0.0; correct=0; n=0
    for xb,yb in loader:
        xb=xb.to(device); yb=yb.to(device)
        optimizer.zero_grad()
        out=model(xb); loss=criterion(out,yb)
        loss.backward(); optimizer.step()
        run_loss+=loss.item()*xb.size(0)
        correct+=(out.argmax(1)==yb).sum().item(); n+=xb.size(0)
    return (run_loss/n, correct/n) if n else (0.0,0.0)

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval(); run_loss=0.0; correct=0; n=0
    for xb,yb in loader:
        xb=xb.to(device); yb=yb.to(device)
        out=model(xb); loss=criterion(out,yb)
        run_loss+=loss.item()*xb.size(0)
        correct+=(out.argmax(1)==yb).sum().item(); n+=xb.size(0)
    return (run_loss/n, correct/n) if n else (0.0,0.0)

def unfreeze_one_layer(model):
    for name,module in reversed(list(model.named_modules())):
        params=list(module.parameters(recurse=False))
        if not params: continue
        if any(not p.requires_grad for p in params):
            for p in params: p.requires_grad=True
            print(f"   Unfroze module: {name or 'root'}")
            return True
    print("   No frozen layers left."); return False


In [ ]:
# Cell 7 — Train loop + run + save outputs to /kaggle/working
os.makedirs("/kaggle/working/fer_primary", exist_ok=True)
os.makedirs("/kaggle/working/fer_comparison", exist_ok=True)

def train_model(model, model_name, out_dir, epochs=20, lr=1e-3):
    criterion=nn.CrossEntropyLoss()
    optimizer,scheduler=get_optimizer_and_scheduler(model,lr)
    best=0.0; no_improve=0; patience_unfreeze=3; history=[]
    print(f"\n{'='*50}\nTraining: {model_name}\n{'='*50}")
    for epoch in range(1,epochs+1):
        tr_loss,tr_acc=train_one_epoch(model,train_loader,optimizer,criterion)
        va_loss,va_acc=validate(model,val_loader,criterion)
        scheduler.step(va_acc)
        lr_now=optimizer.param_groups[0]["lr"]
        history.append(dict(epoch=epoch,train_loss=tr_loss,train_acc=tr_acc,
                            val_loss=va_loss,val_acc=va_acc,lr=lr_now))
        print(f"Epoch {epoch:02d}/{epochs} | tr_loss {tr_loss:.4f} tr_acc {tr_acc:.4f} "
              f"| va_loss {va_loss:.4f} va_acc {va_acc:.4f} | lr {lr_now:.6f}")
        if va_acc>best:
            best=va_acc; no_improve=0
            torch.save(model.state_dict(), f"{out_dir}/best_model.pth")
            print(f"   Best saved! val_acc={best:.4f}")
        else:
            no_improve+=1
            if no_improve>=patience_unfreeze:
                print(f"   No improve {patience_unfreeze} epochs -> unfreezing...")
                if unfreeze_one_layer(model):
                    optimizer=optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),
                                          lr=lr_now,weight_decay=1e-4)
                    scheduler=optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="max",factor=0.5,patience=2)
                no_improve=0
    print(f"\n{model_name} done. Best val_acc={best:.4f}")
    hist=pd.DataFrame(history)
    hist.to_csv(f"{out_dir}/training_log.csv", index=False)
    with open(f"{out_dir}/config.json","w") as f:
        json.dump(dict(arch=model_name,epochs=epochs,lr=lr,image_size=224,
                       batch_size=BATCH,seed=SEED,classes=CLASSES,best_val_acc=best), f, indent=2)
    return hist

EPOCHS = 20   # GPU: fine. Lower to 5 for a quick first pass.
hist_primary    = train_model(primary_model,    "efficientnet_b2",       "/kaggle/working/fer_primary",    epochs=EPOCHS)
hist_comparison = train_model(comparison_model, "mobilenetv3_large_100", "/kaggle/working/fer_comparison", epochs=EPOCHS)

# copy label map next to checkpoints
for d in ["/kaggle/working/fer_primary","/kaggle/working/fer_comparison"]:
    with open(f"{d}/label_map.json","w") as f: json.dump(CLASS_TO_IDX,f,indent=2)
print("\nAll done. Outputs in /kaggle/working/ (Output tab).")


In [ ]:
# Cell 8 — Plot training curves
fig,ax=plt.subplots(1,2,figsize=(13,4))
for h,name in [(hist_primary,"EfficientNet-B2"),(hist_comparison,"MobileNetV3")]:
    ax[0].plot(h["epoch"],h["val_acc"],marker="o",label=name)
    ax[1].plot(h["epoch"],h["val_loss"],marker="o",label=name)
ax[0].set_title("Validation accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].set_title("Validation loss");     ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig("/kaggle/working/training_curves.png", dpi=120); plt.show()
